# Controlled radial ranking: hand-positioned observations

Select **GPU → Run all**. No planning or collected trajectories are used. The goal stays at **(50,42)**, wall x=32 and doorway y=30. We create one fresh goal reference and a new identity-initialized adapter.

Training directions are 24 rays every15°. Training radii are **2,6,10,14,18,22,26,30,34**. Four cumulative stages train through radius **6 → 14 → 22 → 34**. Each stage keeps the earlier positions and continues the adapter, with fresh Adam and **500 ranking-only updates**. Invalid wall/boundary positions are omitted, never clamped.

The fixed test set contains unseen radial midpoints, withheld directions halfway between training rays, and interpolation/offset probes. Heatmaps cover the full right compartment to x≈60. Results go to a new timestamped `radial_ranking_v1` folder.


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, shutil, signal, subprocess, sys, time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
DRIVE = Path('/content/drive/MyDrive')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = DRIVE / 'datasets/curriculum_cost_adapter/radial_ranking_v1' / RUN_ID
CACHE = Path('/content/radial-ranking-cache') / RUN_ID
REPO = Path('/content/i-jepa-wm-radial-ranking')
REPO_URL = 'https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT = None
DINO_ROOT = Path('/content/dino_wm')
DINO_COMMIT = '0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE = DRIVE / 'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
ENCODER = Path('/content/radial-ranking-artifacts/encoder.pth.tar')
ENCODER_SHA = '3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'
assert ENCODER_DRIVE.is_file(), f'Missing encoder: {ENCODER_DRIVE}'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8*1024*1024), b''): h.update(block)
    return h.hexdigest()

def shell(command, cwd=None):
    command=list(map(str,command)); print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout: print(line,end='',flush=True)
            returncode=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.terminate()
            raise
    if returncode: raise subprocess.CalledProcessError(returncode,command)

print('GPU:', torch.cuda.get_device_name())
print('New Drive results:', OUTPUT)
print('Local feature cache:', CACHE)


## Frozen encoder and renderer

Only the existing I-JEPA encoder checkpoint is needed. Old adapter weights, experience archives and world-model checkpoints are not used.


In [ ]:
if not REPO.exists():
    shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
assert SOURCE_COMMIT, 'Use the published notebook with a verified source pin.'
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow'])
if not DINO_ROOT.exists(): shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ENCODER.parent.mkdir(parents=True,exist_ok=True)
if not ENCODER.exists() or sha256(ENCODER)!=ENCODER_SHA: shutil.copy2(ENCODER_DRIVE,ENCODER)
assert sha256(ENCODER)==ENCODER_SHA, 'Unexpected encoder weights.'
assert shutil.disk_usage('/content').free > 7*2**30, 'At least 7 GiB free local disk is needed for the float32 feature cache.'
print('Ready. No world-model checkpoint or previous experiment folder is required.')

def stage_command(operation,index=None):
    args=[sys.executable,'-m','src.mylonas_ijepa_wm.radial_experiment',operation,
          '--output-dir',OUTPUT,'--cache-dir',CACHE,'--device','cuda']
    if operation=='prepare': args+=['--encoder-checkpoint',ENCODER,'--dino-wm-root',DINO_ROOT]
    if index is not None: args+=['--stage-index',str(index)]
    return args


## Encode all fixed positions once

About 4,668 unique images are freshly rendered and encoded in batches of eight. The **float32 probe cache is about5.7GiB and stays on local Colab disk**, avoiding repeated encoder runs. Compact results, training features and the fresh goal image/tokens are saved in Drive.

Pre-encoding a position does not make it a training input: each stage's optimizer receives only its explicitly allowed radial training positions. Midpoints and withheld directions never enter fitting. Farther training positions enter fitting only when their stage is reached.

The encoder is the main cost of this notebook. It runs once for all four stages.


In [ ]:
started=time.monotonic()
shell(stage_command('prepare'),cwd=REPO)
design=json.loads((OUTPUT/'design.json').read_text())
config=json.loads((OUTPUT/'config.json').read_text())
print(f'Fresh encoding took {(time.monotonic()-started)/60:.1f} minutes.')
print('Unique encoded images:',len(design['points']))
print('Stage limits and cumulative training positions:',[(s['outer_radius'],len(s['training_ids'])) for s in design['stages']])
print('Matched offset families:',len(design['matched_family_ids']))
print('Fitting:',config['fitting'],'| ranking only; no anchor loss')
print('Fresh goal image:')
from IPython.display import display,Image
display(Image(filename=str(OUTPUT/'goal.png')))


## Four cumulative fits with results after each update

Pairs compare different training radii, including points on different rays: farther must have higher adapted MSE. Equal-radius costs are not constrained. The architecture, softplus ranking loss and no-anchor setting remain as before.

**Reading the results:**
- Heatmaps use independent unclipped raw/adapted scales. White points are training positions; dashed circle is the current radius.
- Radius plots go **outward**, so costs should rise. Interpolation plots go **toward the goal**, so costs should fall.
- `inside` comparisons have both endpoints within the current radius; `outside` have both beyond it; `crossing` straddle it.
- `heldout` comparisons exclude positions or rendered images overlapping any offered training input.
- Offset tests use a matched family set at every offset and average ± sides within each family. The offered-endpoint subset tests local behavior around that stage's training pairs; the second table also includes future radii.
- Flat curves are counted separately, not treated as useful monotonicity. Fractions are descriptive; nearby samples are not independent trials.
- Loss plots include a stable measurement over **all eligible training pairs**, alongside noisy sampled-batch losses.

Rerunning this stage cell within the same runtime reuses validated completed stages. Rerunning the setup cell creates a fresh timestamped experiment. If the runtime is lost, the local token cache is lost too; existing Drive results remain available.


In [ ]:
def pct(value): return 'N/A' if value is None else f'{value:.1%}'

def show_report(report):
    t=report['training']
    print(f"\nSTAGE {report['stage']} | max radius {report['outer_radius']} | {report['training_positions']} training positions")
    print('Initialization:',report['initialization'],'| eligible training pairs:',t['eligible_pairs'])
    print('All-training-pair ranking accuracy:',pct(t['rank_accuracy_before']),'→',pct(t['rank_accuracy_after']))
    print('All-training-pair loss:',t['full_rank_loss_before'],'→',t['history'][-1]['full_rank_loss'])
    print('Raw/adapted goal self-cost:',report['raw_goal_self_cost'],report['adapted_goal_self_cost'])
    print('New primitive actions:',report['primitive_interactions'])
    print('\nDirectional comparisons: uphill means wrong direction while moving TOWARD the goal')
    print(f"{'directions':>20} {'region':>10} {'method':>8} {'steps':>7} {'uphill':>9} {'heldout up / steps':>22} {'ties':>7}")
    for kind,regions in report['generalization'].items():
        for region,methods in regions.items():
            for method,m in methods.items():
                print(f"{kind:>20} {region:>10} {method:>8} {m['comparisons']:>7} {pct(m['up_fraction']):>9} "
                      f"{m['heldout_up_steps']:>9}/{m['heldout_comparisons']:<11} {m['tie_steps']:>7}")
    for key,label in [('offset_fitted_centers','OFFERED TRAINING ENDPOINTS'),('offset_summary','FIXED COHORT ACROSS ALL RADII')]:
        print('\nMATCHED OFFSET TEST:',label)
        print(f"{'offset':>7} {'families':>9} {'method':>8} {'endpoint OK':>12} {'monotone':>10} {'uphill':>9} {'flat':>8}")
        for row in report[key]:
            for method in ('raw','adapted'):
                m=row[method]
                print(f"{row['offset']:>7g} {row['families']:>9} {method:>8} {pct(m['endpoint_correct']):>12} "
                      f"{pct(m['informative_monotone']):>10} {pct(m['up_fraction']):>9} {pct(m['flat']):>8}")
    unseen=sum(report['exposure']['heldout_from_offered_training'])
    print(f'Coordinates with no coordinate/image overlap with offered training: {unseen}/{len(design["points"])}')
    print('Nearest-training distances and per-profile exposure flags are saved in report.json.')

reports=[]
for index in (1,2,3,4):
    started=time.monotonic()
    shell(stage_command('stage',index),cwd=REPO)
    folder=OUTPUT/f'stages/s{index}'
    report=json.loads((folder/'report.json').read_text()); reports.append(report)
    show_report(report)
    print(f'Stage took {(time.monotonic()-started)/60:.1f} minutes.')
    for name in ('heatmaps','radial_profiles','offset_summary','interpolation_examples','training_loss'):
        display(Image(filename=str(folder/'figures'/f'{name}.png')))


## Compare the four stages

Use heldout directions to assess whether learning extends between training rays. Use the beyond-radius results to see whether it extrapolates before those distances enter training. Compare raw/adapted errors with their denominators, not training loss alone.


In [ ]:
print('SUMMARY: uphill fractions on genuinely withheld-direction comparisons')
print(f"{'stage':>6} {'R':>5} {'train':>7} {'inside raw':>12} {'inside adapt':>13} {'outside raw':>12} {'outside adapt':>14}")
for r in reports:
    g=r['generalization']['withheld_direction']
    print(f"{r['stage']:>6} {r['outer_radius']:>5} {r['training_positions']:>7} "
          f"{pct(g['inside']['raw']['heldout_up_fraction']):>12} {pct(g['inside']['adapted']['heldout_up_fraction']):>13} "
          f"{pct(g['outside']['raw']['heldout_up_fraction']):>12} {pct(g['outside']['adapted']['heldout_up_fraction']):>14}")
print('\nInside/outside populations change as the training radius expands; counts are in the detailed reports.')
print('No outside-ray comparisons remain at R34 because the fixed directional probes stop at radius34; N/A is not zero error.')
print('All results and figures:',OUTPUT)


## Return the results

Download the executed notebook (**File → Download → Download .ipynb**) and attach it here. Each stage saves its checkpoint, fitting metadata, numeric report and five PNG/PDF figures in Drive. No simulator actions or planning episodes occur.

This experiment tests spatial generalization under explicit distance supervision for one fixed goal. Transfer to other goals and planning performance remain separate questions.
